In [1]:
import pandas as pd
from sklearn import datasets
    
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset

from sklearn.metrics import mean_squared_log_error
import numpy as np

# Define evaluation metrics
def rmsle(y_true, y_pred):
    # Clip predictions to avoid log(0)
    y_pred = np.maximum(0, y_pred)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

### Load dataset

In [2]:
PREPARED_TRAIN_DATA_PATH = '../data/prepared/df_training.parquet'
PREPARED_VALIDATION_DATA_PATH = '../data/prepared/df_validation.parquet'

train = pd.read_parquet(PREPARED_TRAIN_DATA_PATH)
validation = pd.read_parquet(PREPARED_VALIDATION_DATA_PATH)

### Evidently report

In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960000 entries, 0 to 959999
Data columns (total 5 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   credit_score            960000 non-null  float64
 1   customer_feedback_good  960000 non-null  float64
 2   annual_income           960000 non-null  float64
 3   health_score            960000 non-null  float64
 4   premium_amount          960000 non-null  float64
dtypes: float64(5)
memory usage: 36.6 MB


In [4]:
# data labelling
target = "premium_amount"
numerical_features = ["credit_score", "customer_feedback_good", "annual_income", "health_score"]

In [5]:
import pickle
MODEL_PATH = '../models/xgb_model.pickle'
with open (MODEL_PATH, 'rb') as f_in:
    loaded_model = pickle.load(f_in)

In [6]:
import xgboost as xgb

X_train = train[numerical_features]
y_train = train[target].values

X_validation = validation[numerical_features]
y_validation = validation[target].values

xgb_train = xgb.DMatrix(X_train, label=y_train)
xgb_valid = xgb.DMatrix(X_validation, label=y_validation)

In [7]:
train_preds = loaded_model.predict(xgb_train)
train['prediction'] = train_preds

validation_preds = loaded_model.predict(xgb_valid)
validation['prediction'] = validation_preds

In [8]:
print(rmsle(train.premium_amount, train.prediction))
print(rmsle(validation.premium_amount, validation.prediction))

1.1548608900858948
1.1590481730987847


In [10]:
# Map the column types
schema = DataDefinition(
    numerical_columns=numerical_features,
)

In [11]:
# Create Evidently Datasets to work with
eval_data_1 = Dataset.from_pandas(
    pd.DataFrame(train),
    data_definition=schema
)

eval_data_2 = Dataset.from_pandas(
    pd.DataFrame(validation),
    data_definition=schema
)

In [19]:
# Get report
report = Report([

    DataSummaryPreset(),
    DataDriftPreset(),

])

my_eval = report.run(eval_data_1, eval_data_2)



In [22]:
# save the report
my_eval.save_html("evidently_report.html")